# Notebook 31 — FT-Transformer arm: does pruning-induced collapse replicate without a locality prior?

The CNN1D results implicate its convolutional locality prior as a contributor to decision-layer
reallocation. This notebook runs the paired-seed protocol of Notebook 11 and the mechanism spot-suite
of Notebooks 12/09 on the attention architecture `FTTransformerLite`, which has no locality prior.

**Gate (stated before running):** the collapse story replicates on the transformer if (i) mean paired
prune80 macro-F1 loss exceeds 0.15 across five independent baseline-mask pairs, (ii) at least four
classes are materially affected in three or more seeds, (iii) leakage-safe probe AUC stays at or above
0.85 for every collapsed class (representation survives), and (iv) a refit dense head recovers at least
half of the macro-F1 loss. Any outcome is reported; a non-replication is itself a boundary-condition
finding and changes the paper's framing rather than being a failure.

**Pruner note.** The archived pruner covers only `nn.Linear` and `nn.Conv1d` modules. The transformer
stores its attention projections as a raw parameter (`in_proj_weight`) on `nn.MultiheadAttention`,
which that pruner would leave dense, so "prune80" would silently be ~50% whole-model sparsity. This
notebook uses a transformer-aware pruner that includes attention projections, otherwise line-for-line
identical to `src.compression.prune_and_finetune` (layer-wise L1, gradient-masked fine-tuning, final
hard mask). Achieved sparsity is measured and asserted, not assumed.

Checkpoints are saved per seed and the loop resumes from existing checkpoints, so a runtime
disconnect never repeats a finished seed. GPU runtime required.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)

import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, train_model, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET, ARCH = 'ciciot2023', 'ft_transformer'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
print('seeds:', SEEDS, '| anchor:', ANCHOR, '| device:', DEVICE)

In [ ]:
# Data and the frozen primary split (identical to every other notebook)
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes | {len(feat_cols)} features')

In [ ]:
# Recover the anchor transformer's architecture kwargs from its checkpoint.
# d_token and n_layers are determined by tensor shapes; n_heads is not (it does not change
# parameter shapes), so it is identified as the candidate that reproduces the trained model's
# validation macro-F1 - only the true head count computes the trained function.
ckpt = torch.load(PATHS.model(DATASET, ARCH, 'M0', ANCHOR), map_location='cpu', weights_only=False)
sd = ckpt['state_dict']
d_token = int(sd['tokenizer.weight'].shape[0])
n_layers = len({k.split('.')[2] for k in sd if k.startswith('encoder.layers.')})
print(f'inferred d_token={d_token}, n_layers={n_layers}')


cands = []
for h in (4, 2, 8, 1):
    if d_token % h: continue
    kw = {'d_token': d_token, 'n_heads': h, 'n_layers': n_layers}
    try:
        m, le, scaler, fc = load_anchor(DATASET, ARCH, 'M0', ANCHOR, arch_kwargs=kw)
        yv, pv, _ = predict(m, df, splits, le, scaler, fc, which='val')
        cands.append((f1_score(yv, pv, average='macro'), h))
        print(f'  n_heads={h}: validation macro-F1 = {cands[-1][0]:.4f}')
    except Exception as exc:
        print(f'  n_heads={h}: failed ({type(exc).__name__})')
best_f1, best_h = max(cands)
ARCH_KW = {'d_token': d_token, 'n_heads': best_h, 'n_layers': n_layers}
assert best_f1 >= 0.45, f'anchor transformer sanity gate failed: best val macro-F1 {best_f1:.3f}'
print('ARCH_KW =', ARCH_KW)

# Architecture gate against the archived CNN/MLP validation bands
gate = pd.read_csv(OUT / 'validation_architecture_gate.csv')
for a in ('cnn1d', 'mlp'):
    g = gate[(gate.arch == a) & (gate.status == 'ok')]['validation_macro_f1']
    print(f'  {a}: validation band {g.min():.3f}-{g.max():.3f}')
print(f'  transformer anchor: {best_f1:.3f}')

In [ ]:
# Transformer-aware pruner: identical to src.compression.prune_and_finetune except that the
# prunable set includes nn.MultiheadAttention.in_proj_weight. Layer-wise L1 at `amount` per tensor.
def prunable(model):
    out = []
    for mod in model.modules():
        if isinstance(mod, (nn.Linear, nn.Conv1d)):
            out.append((mod, 'weight'))
        elif isinstance(mod, nn.MultiheadAttention) and getattr(mod, 'in_proj_weight', None) is not None:
            out.append((mod, 'in_proj_weight'))
    return out

def magnitude_prune_ext(model, amount):
    m = copy.deepcopy(model)
    for mod, name in prunable(m):
        prune.l1_unstructured(mod, name=name, amount=amount)
        prune.remove(mod, name)
    return m

def sparsity_report(model):
    z = n = 0
    for mod, name in prunable(model):
        w = getattr(mod, name); z += int((w == 0).sum()); n += w.numel()
    tot = sum(p.numel() for p in model.parameters()); ztot = sum(int((p == 0).sum()) for p in model.parameters())
    return {'prunable_sparsity': z / n, 'whole_model_sparsity': ztot / tot, 'prunable_params': n, 'total_params': tot}

def prune_and_finetune_ext(anchor, seed, amount, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler)
    Xtr, ytr = t['train']
    model = magnitude_prune_ext(anchor, amount).to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    prune{int(amount*100)} ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model):
            getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    rep = sparsity_report(model)
    assert abs(rep['prunable_sparsity'] - amount) < 0.02, rep
    return model.eval(), le, scaler, rep

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)
print('pruner ready')

In [ ]:
# Paired independent seeds with resume. Cells: M0_tf_paired, prune50_tf_paired, prune80_tf_paired.
baseline_val, baseline_test, comp_test, macro, pruned_models, sparsity_rows = {}, {}, {}, [], {}, []
for seed in SEEDS:
    print(f'\n===== seed {seed} =====')
    p_m0 = PATHS.model(DATASET, ARCH, 'M0_tf_paired', seed)
    if os.path.exists(p_m0):
        m0, le, scaler, fc = load_anchor(DATASET, ARCH, 'M0_tf_paired', seed, arch_kwargs=ARCH_KW); print('  loaded baseline')
    else:
        m0, info = train_model(ARCH, df, DATASET, splits, seed, epochs=40, patience=6, batch_size=4096, lr=1e-3,
                               compression='M0_tf_paired', arch_kwargs=ARCH_KW, save=True, verbose=True)
        le, scaler = info['label_encoder'], info['scaler']
    yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val')
    yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
    baseline_val[seed] = per_class_recall_table(yv, pv, le).set_index('label')['recall']
    baseline_test[seed] = per_class_recall_table(yt, pt, le).set_index('label')['recall']
    macro.append({'seed': seed, 'cell': 'M0', 'test_macro_f1': f1_score(yt, pt, average='macro')})

    for amount, cell in ((0.50, 'prune50'), (0.80, 'prune80')):
        p_c = PATHS.model(DATASET, ARCH, f'{cell}_tf_paired', seed)
        if os.path.exists(p_c):
            mp = M.build(ARCH, len(feat_cols), len(le.classes_), **ARCH_KW).to(DEVICE)
            mp.load_state_dict(torch.load(p_c, map_location=DEVICE, weights_only=False)['state_dict']); mp.eval()
            rep = sparsity_report(mp); print(f'  loaded {cell}')
        else:
            mp, _, _, rep = prune_and_finetune_ext(m0, seed, amount, verbose=True)
            save_ckpt(mp, le, scaler, p_c); print('  saved', p_c)
        sparsity_rows.append({'seed': seed, 'cell': cell, **rep})
        yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
        comp_test[(seed, cell)] = per_class_recall_table(yt, pc, le).set_index('label')['recall']
        macro.append({'seed': seed, 'cell': cell, 'test_macro_f1': f1_score(yt, pc, average='macro')})
        if cell == 'prune80': pruned_models[seed] = mp
print('\nall seeds complete')

In [ ]:
# Aggregate: schema-identical to Notebook 11 outputs, prefixed transformer_
tiers = assign_validation_tiers(pd.DataFrame(baseline_val))
rows = []
for (seed, cell), rc in comp_test.items():
    r0 = baseline_test[seed]
    for cls in r0.index.intersection(rc.index):
        loss = float(r0.loc[cls] - rc.loc[cls]); band = float(tiers.loc[cls, 'validation_2sd_band'])
        rows.append({'seed': seed, 'cell': cell, 'class': cls, 'M0_test_recall': float(r0.loc[cls]),
                     'compressed_test_recall': float(rc.loc[cls]), 'recall_loss': loss,
                     'validation_tier': tiers.loc[cls, 'validation_tier'], 'validation_2sd_band': band,
                     'crosses_validation_band': bool(loss > band), 'practically_material': bool(loss >= PRACTICAL_LOSS),
                     'material_and_beyond_band': bool((loss > band) and (loss >= PRACTICAL_LOSS))})
paired = pd.DataFrame(rows); paired.to_csv(OUT / 'transformer_paired_seed_per_class_effects.csv', index=False)
summary = paired.groupby(['cell', 'class']).agg(mean_recall_loss=('recall_loss', 'mean'), sd_recall_loss=('recall_loss', 'std'),
          affected_frequency=('material_and_beyond_band', 'mean'), n=('seed', 'nunique')).reset_index()
summary.to_csv(OUT / 'transformer_paired_seed_per_class_summary.csv', index=False)
mdf = pd.DataFrame(macro); mdf.to_csv(OUT / 'transformer_paired_seed_macro_f1_wide.csv', index=False)
msum = mdf.groupby('cell')['test_macro_f1'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
msum.columns = ['cell', 'n', 'mean', 'sd', 'min', 'max']; msum.to_csv(OUT / 'transformer_paired_seed_macro_f1_summary.csv', index=False)
pd.DataFrame(sparsity_rows).to_csv(OUT / 'transformer_paired_seed_sparsity.csv', index=False)

# mask Jaccard over the prunable set actually pruned (attention projections included)
def flat_mask(model): return np.concatenate([(getattr(mod, n).detach().cpu().numpy() != 0).ravel() for mod, n in prunable(model)])
jac = []; ks = sorted(pruned_models)
for i, a in enumerate(ks):
    for b in ks[i+1:]:
        ma, mb = flat_mask(pruned_models[a]), flat_mask(pruned_models[b])
        jac.append({'seed_a': a, 'seed_b': b, 'global_nonzero_mask_jaccard': float(np.logical_and(ma, mb).sum() / np.logical_or(ma, mb).sum())})
jdf = pd.DataFrame(jac); jdf.to_csv(OUT / 'transformer_paired_seed_mask_jaccard.csv', index=False)
tiers.to_csv(OUT / 'transformer_paired_validation_defined_tiers.csv')
print(msum.round(4).to_string(index=False)); print()
print('mask Jaccard mean:', round(jdf.global_nonzero_mask_jaccard.mean(), 4))
print(pd.DataFrame(sparsity_rows).groupby('cell')[['prunable_sparsity', 'whole_model_sparsity']].mean().round(4))
print(summary[summary.cell == 'prune80'].sort_values('mean_recall_loss', ascending=False).head(12).round(4).to_string(index=False))

In [ ]:
# Mechanism spot-suite on the anchor-seed pair: leakage-safe probes + dense-head refit.
# (No BatchNorm control: the transformer uses LayerNorm, so the stale-statistics hypothesis does not apply.)
seed = ANCHOR
m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_tf_paired', seed, arch_kwargs=ARCH_KW)
p80 = pruned_models[seed]
p80_sum = summary[(summary.cell == 'prune80') & (summary.affected_frequency >= 0.6)]
collapsed = list(p80_sum['class'])
print(f'{len(collapsed)} classes materially affected in >=3/5 seeds')

fit = {}
for which in ('train', 'val'):
    f0, _, y = EXP.extract_features(m0, df, splits, scaler, feat_cols, le, which=which)
    fc, _, _ = EXP.extract_features(p80, df, splits, scaler, feat_cols, le, which=which)
    fit[which] = (f0, fc, y)
f0_te, _, y_te = EXP.extract_features(m0, df, splits, scaler, feat_cols, le, which='test')
fc_te, _, _ = EXP.extract_features(p80, df, splits, scaler, feat_cols, le, which='test')
f0_fit = np.concatenate([fit['train'][0], fit['val'][0]]); fc_fit = np.concatenate([fit['train'][1], fit['val'][1]])
y_fit = np.concatenate([fit['train'][2], fit['val'][2]])
rng = np.random.default_rng(seed); keep = []
for c in np.unique(y_fit):
    idx = np.where(y_fit == c)[0]
    keep.append(rng.choice(idx, 8000, replace=False) if len(idx) > 8000 else idx)
keep = np.concatenate(keep); f0_fit, fc_fit, y_fit = f0_fit[keep], fc_fit[keep], y_fit[keep]

def probe(f_fit, f_te, c):
    yb, yt_ = (y_fit == c).astype(int), (y_te == c).astype(int)
    if yb.sum() < 5 or yt_.sum() < 2: return np.nan
    clf = LogisticRegression(max_iter=500, C=1.0, class_weight='balanced').fit(f_fit, yb)
    return roc_auc_score(yt_, clf.predict_proba(f_te)[:, 1])
prows = []
for cname in collapsed:
    c = int(np.where(le.classes_ == cname)[0][0]); a0, ac = probe(f0_fit, f0_te, c), probe(fc_fit, fc_te, c)
    prows.append({'label': cname, 'auc_M0': a0, 'auc_prune80': ac, 'auc_drop': a0 - ac})
pdf = pd.DataFrame(prows); pdf.to_csv(OUT / 'transformer_leakage_safe_probe.csv', index=False)
print(pdf.round(4).to_string(index=False))

Lval, yval, Ltest, ytest = mitigate.refit_head(p80, df, splits, scaler, feat_cols, le, epochs=15, lr=1e-2, batch_size=4096, seed=seed)
pred_refit = np.asarray(Ltest).argmax(1)
refit_f1 = f1_score(ytest, pred_refit, average='macro')
cal = calibration_summary(torch.softmax(torch.tensor(Ltest), dim=1).numpy(), ytest); cal.insert(0, 'cell', 'transformer_prune80_head_refit')
cal.to_csv(OUT / 'transformer_head_refit_calibration.csv', index=False)
print(f'\nhead refit macro-F1: {refit_f1:.4f}'); print(cal.round(4).to_string(index=False))

In [ ]:
# Gate verdict - written whichever way it falls
m0_mean = float(msum.loc[msum.cell == 'M0', 'mean']); p80_mean = float(msum.loc[msum.cell == 'prune80', 'mean'])
loss = m0_mean - p80_mean
n_aff = int((summary[(summary.cell == 'prune80')]['affected_frequency'] >= 0.6).sum())
min_auc = float(pdf['auc_prune80'].min()) if len(pdf) else float('nan')
recovery = (refit_f1 - p80_mean) / loss if loss > 0 else float('nan')
verdict = pd.DataFrame([
 {'criterion': 'i_mean_prune80_macro_f1_loss_gt_0.15', 'value': round(loss, 4), 'pass': loss > 0.15},
 {'criterion': 'ii_classes_affected_in_ge3of5_seeds_ge_4', 'value': n_aff, 'pass': n_aff >= 4},
 {'criterion': 'iii_min_probe_auc_collapsed_ge_0.85', 'value': round(min_auc, 4), 'pass': bool(min_auc >= 0.85)},
 {'criterion': 'iv_refit_recovers_ge_half_of_loss', 'value': round(recovery, 4), 'pass': bool(recovery >= 0.5)},
])
verdict['arch_kwargs'] = str(ARCH_KW)
print(verdict.to_string(index=False))
print('\nCOLLAPSE STORY REPLICATES ON TRANSFORMER:', bool(verdict['pass'].all()))
verdict.to_csv(OUT / 'transformer_gate_verdict.csv', index=False)
write_json(OUT / 'transformer_arm_environment.json', {'arch_kwargs': ARCH_KW, 'seeds': SEEDS, 'practical_loss': PRACTICAL_LOSS,
                                                     'pruner': 'layer-wise L1 incl. attention in_proj', 'environment': environment_record()})

In [ ]:
# --- Commit + push (identity explicit; notebook outputs stripped before staging) ---
import subprocess, shutil, glob
# --- Branch guard: this notebook belongs to the diagnostic paper on main, not the SABER branch ---
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` in the repo first (SABER work stays on saber-ids-method)'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
# strip outputs from THIS notebook only, then stage ONLY this notebook's own files
_own = 'notebooks/31_transformer_arm_paired_seeds.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/transformer_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 31: FT-Transformer arm - paired seeds with attention-inclusive pruner, probes, head refit, gate verdict'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)